## 1. Setup & Config

In [ ]:
%%capture
!pip install transformers peft scikit-learn scipy matplotlib sentencepiece "torchao>=0.16.0"

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from typing import Any, Dict, Optional, Sequence, Tuple, List
import torch.nn as nn
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import RobertaModel, DebertaV2Model, RobertaTokenizer, DebertaV2Tokenizer
from transformers import get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model
import random
import matplotlib.pyplot as plt
import glob
from scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression
import torch.nn.functional as F

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive"
DATA = f"{BASE}/Dataset 1"
MDL  = f"{BASE}/Analysis Subtask 1/models_dump"
RES  = f"{BASE}/Analysis Subtask 1/results7"
os.makedirs(MDL, exist_ok=True); os.makedirs(RES, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 2. Data Preparation

In [ ]:
def build_temporal_features(df, train_ref):
    df = df.copy(); df["_ts"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values(["user_id","_ts"]).reset_index(drop=True)
    if train_ref is not None:
        fb = (train_ref.groupby("user_id")[["valence","arousal"]].mean()
              .rename(columns={"valence":"_fb_v","arousal":"_fb_a"}))
        df = df.join(fb, on="user_id")
    else: df["_fb_v"]=0.0; df["_fb_a"]=0.0
    df["prev_v"]=df.groupby("user_id")["valence"].shift(1)
    df["prev_a"]=df.groupby("user_id")["arousal"].shift(1)
    df["_prev_ts"]=df.groupby("user_id")["_ts"].shift(1)
    df["is_first"]=df["prev_v"].isna().astype(float)
    df["prev_v"]=df["prev_v"].fillna(df["_fb_v"]); df["prev_a"]=df["prev_a"].fillna(df["_fb_a"])
    df["delta_hours"]=(df["_ts"]-df["_prev_ts"]).dt.total_seconds()/3600.0
    df["delta_hours"]=df["delta_hours"].fillna(0.0)
    df["log_delta"]=np.log1p(df["delta_hours"])
    mu,sig=df["log_delta"].mean(),df["log_delta"].std()+1e-9
    df["log_delta"]=(df["log_delta"]-mu)/sig
    df["is_words_f"]=df["is_words"].astype(float) if "is_words" in df.columns else 0.0
    df["phase_idx"]=df["collection_phase"].astype(int) if "collection_phase" in df.columns else 0
    df["delta_time"]=df["log_delta"]
    return df

def build_temporal_features_test(test_df, train_ref):
    df = test_df.copy(); df["_ts"]=pd.to_datetime(df["timestamp"])
    df=df.sort_values(["user_id","_ts"]).reset_index(drop=True)
    fb=(train_ref.groupby("user_id")[["valence","arousal"]].mean()
        .rename(columns={"valence":"_fb_v","arousal":"_fb_a"}))
    df=df.join(fb, on="user_id")
    df["_fb_v"]=df["_fb_v"].fillna(train_ref["valence"].mean())
    df["_fb_a"]=df["_fb_a"].fillna(train_ref["arousal"].mean())
    df["prev_v"],df["prev_a"],df["is_first"]=df["_fb_v"],df["_fb_a"],1.0
    df["_prev_ts"]=df.groupby("user_id")["_ts"].shift(1)
    df["delta_hours"]=(df["_ts"]-df["_prev_ts"]).dt.total_seconds()/3600.0
    df["delta_hours"]=df["delta_hours"].fillna(0.0)
    df["log_delta"]=np.log1p(df["delta_hours"])
    mu,sig=df["log_delta"].mean(),df["log_delta"].std()+1e-9
    df["log_delta"]=(df["log_delta"]-mu)/sig
    df["delta_time"]=df["log_delta"]
    df["is_words_f"]=df["is_words"].astype(float) if "is_words" in df.columns else 0.0
    df["phase_idx"]=df["collection_phase"].astype(int) if "collection_phase" in df.columns else 0
    return df

def temporal_user_split(df, val_ratio=0.2):
    tr, va = [], []
    for uid, grp in df.groupby("user_id"):
        grp=grp.sort_values("timestamp").reset_index(drop=True); n=len(grp)
        if n==1: tr.append(grp); continue
        nv=max(1,int(np.floor(n*val_ratio)))
        tr.append(grp.iloc[:-nv]); va.append(grp.iloc[-nv:])
    return pd.concat(tr).reset_index(drop=True), pd.concat(va).reset_index(drop=True)

In [ ]:
class VADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, has_labels=True):
        self.df=df.reset_index(drop=True); self.tok=tokenizer
        self.max_len=max_len; self.has_labels=has_labels
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]; text=str(row["text"]) if pd.notna(row["text"]) else ""
        enc=self.tok(text, truncation=True, padding="max_length",
                     max_length=self.max_len, return_tensors="pt")
        item={"input_ids":enc["input_ids"].squeeze(0),
              "attention_mask":enc["attention_mask"].squeeze(0),
              "token_type_ids":enc.get("token_type_ids",
                               torch.zeros_like(enc["input_ids"])).squeeze(0),
              "user_idx":torch.tensor(int(row["user_idx_mapped"]),dtype=torch.long),
              "prev_v":torch.tensor(float(row["prev_v"]),dtype=torch.float),
              "prev_a":torch.tensor(float(row["prev_a"]),dtype=torch.float),
              "log_delta":torch.tensor(float(row["log_delta"]),dtype=torch.float),
              "delta_time":torch.tensor(float(row["delta_time"]),dtype=torch.float),
              "is_words_f":torch.tensor(float(row["is_words_f"]),dtype=torch.float),
              "is_first":torch.tensor(float(row["is_first"]),dtype=torch.float),
              "phase_idx":torch.tensor(int(row["phase_idx"]),dtype=torch.long),
              "user_id_str":str(int(row["user_id"])),
              "text_id_str":str(int(row["text_id"]))}
        if self.has_labels:
            item["valence"]=torch.tensor(float(row["valence"]),dtype=torch.float)
            item["arousal"]=torch.tensor(float(row["arousal"]),dtype=torch.float)
        return item

## 3. Model Architecture

In [ ]:
class MeanPooling(nn.Module):
    def forward(self, hidden, mask):
        m = mask.unsqueeze(-1).float()
        return torch.sum(hidden * m, 1) / m.sum(1).clamp(min=1e-9)

class TemporalEncoder(nn.Module):
    def __init__(self, phase_emb_dim=8, n_phases=8, out_dim=32):
        super().__init__()
        self.phase_emb = nn.Embedding(n_phases, phase_emb_dim, padding_idx=0)
        self.mlp = nn.Sequential(nn.Linear(5+phase_emb_dim, out_dim), nn.ReLU(),
                                 nn.Linear(out_dim, out_dim))
    def forward(self, prev_v, prev_a, log_delta, is_words, is_first, phase_idx):
        p = self.phase_emb(phase_idx)
        s = torch.stack([prev_v, prev_a, log_delta, is_words, is_first], dim=1)
        return self.mlp(torch.cat([s, p], dim=1))

class RobertaVAModel(nn.Module):
    def __init__(self, n_users, checkpoint="FacebookAI/roberta-large", n_unfreeze_layers=2):
        super().__init__()
        base = RobertaModel.from_pretrained(checkpoint).to(torch.float32)
        cfg = LoraConfig(r=16, lora_alpha=32, target_modules=["query","key","value"],
                         lora_dropout=0.1, bias="none")
        self.encoder = get_peft_model(base, cfg)
        bm = self.encoder.base_model.model; L = len(bm.encoder.layer)
        for i in range(L-n_unfreeze_layers, L):
            for p in bm.encoder.layer[i].parameters(): p.requires_grad = True
        self.encoder.print_trainable_parameters()
        hid = self.encoder.config.hidden_size
        self.user_emb = nn.Embedding(n_users+1, 64, padding_idx=0)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        self.temporal_enc = TemporalEncoder(); self.pooling = MeanPooling()
        self.fusion = nn.Sequential(
            nn.Linear(hid+64+32, 512), nn.GELU(), nn.LayerNorm(512), nn.Dropout(0.15),
            nn.Linear(512, 256), nn.GELU(), nn.LayerNorm(256), nn.Dropout(0.10))
        self.val_head = nn.Sequential(nn.Linear(256,64), nn.GELU(), nn.Linear(64,1))
        self.aro_head = nn.Sequential(nn.Linear(256,64), nn.GELU(), nn.Linear(64,1))
    def forward(self, input_ids, attention_mask, user_idx,
                prev_v, prev_a, log_delta, is_words_f, is_first, phase_idx):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        h = self.pooling(enc.last_hidden_state, attention_mask)
        e = self.user_emb(user_idx)
        c = self.temporal_enc(prev_v, prev_a, log_delta, is_words_f, is_first, phase_idx)
        f = self.fusion(torch.cat([h, e, c], dim=1))
        return self.val_head(f).squeeze(-1), self.aro_head(f).squeeze(-1)

class DebertaVAModel(nn.Module):
    def __init__(self, n_users, checkpoint="microsoft/deberta-v3-large", n_unfreeze_layers=2):
        super().__init__()
        base = DebertaV2Model.from_pretrained(checkpoint).to(torch.float32)
        cfg = LoraConfig(r=16, lora_alpha=32,
                         target_modules=["query_proj","key_proj","value_proj","pos_proj"],
                         lora_dropout=0.1, bias="none")
        self.encoder = get_peft_model(base, cfg)
        bm = self.encoder.base_model.model; L = len(bm.encoder.layer)
        for i in range(L-n_unfreeze_layers, L):
            for p in bm.encoder.layer[i].parameters(): p.requires_grad = True
        self.encoder.print_trainable_parameters()
        hid = self.encoder.config.hidden_size
        self.user_emb = nn.Embedding(n_users+1, 64, padding_idx=0)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        self.temporal_enc = TemporalEncoder(); self.pooling = MeanPooling()
        self.fusion = nn.Sequential(
            nn.Linear(hid+64+32, 512), nn.GELU(), nn.LayerNorm(512), nn.Dropout(0.15),
            nn.Linear(512, 256), nn.GELU(), nn.LayerNorm(256), nn.Dropout(0.10))
        self.val_head = nn.Sequential(nn.Linear(256,64), nn.GELU(), nn.Linear(64,1))
        self.aro_head = nn.Sequential(nn.Linear(256,64), nn.GELU(), nn.Linear(64,1))
    def forward(self, input_ids, attention_mask, token_type_ids, user_idx,
                prev_v, prev_a, log_delta, is_words_f, is_first, phase_idx):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        h = self.pooling(enc.last_hidden_state, attention_mask)
        e = self.user_emb(user_idx)
        c = self.temporal_enc(prev_v, prev_a, log_delta, is_words_f, is_first, phase_idx)
        f = self.fusion(torch.cat([h, e, c], dim=1))
        return self.val_head(f).squeeze(-1), self.aro_head(f).squeeze(-1)

## 4. Training & Evaluation Setup

In [ ]:
def _pearson(x, y):
    x = np.asarray(x, float); y = np.asarray(y, float)
    if x.size < 2: return float("nan"), None
    r, p = pearsonr(x, y); return float(r), float(p)

def _mae(x, y):
    return float(np.nanmean(np.abs(np.asarray(x,float) - np.asarray(y,float))))

def task1_correlation(user_ids, text_ids, predictions, labels) -> Dict[str, Any]:
    user_arr = np.asarray(user_ids)
    preds = np.asarray(predictions, float); labs = np.asarray(labels, float)
    uu = np.unique(user_arr); r_vals, p_vals = [], []
    for u in uu:
        mask = user_arr == u
        if np.sum(mask) < 2: continue
        if np.var(labs[mask]) == 0.0: continue
        if np.var(preds[mask]) == 0.0:
            r_vals.append(0.0); p_vals.append(1e-10); continue
        r, p = _pearson(preds[mask], labs[mask])
        r_vals.append(r); p_vals.append(float(p))
    r_within = float(np.mean(r_vals))
    p_within = float(len(p_vals) / sum(1.0/max(pv,1e-10) for pv in p_vals))
    ump, uml = [], []
    for u in uu:
        mask = user_arr == u
        ump.append(np.nanmean(preds[mask])); uml.append(np.nanmean(labs[mask]))
    r_between, p_between = _pearson(ump, uml)
    mae_within  = float(np.nanmean([_mae(preds[user_arr==u], labs[user_arr==u]) for u in uu]))
    mae_between = _mae(ump, uml)
    z_w, z_b = np.arctanh(r_within), np.arctanh(r_between)
    r_composite = float(np.tanh(0.5*(z_w+z_b)))
    mae_composite = float(np.tanh(0.5*(np.arctanh(mae_within)+np.arctanh(mae_between))))
    return {"r_within": r_within, "p_within": p_within,
            "r_between": r_between, "p_between": p_between,
            "r_composite": r_composite, "mae_within": mae_within,
            "mae_between": mae_between, "mae_composite": mae_composite}

In [ ]:
def make_optimizer(model, lr=3e-5, weight_decay=0.01):
    nd = {"bias","LayerNorm.weight","layer_norm.weight"}
    return AdamW([
        {"params":[p for n,p in model.named_parameters()
                   if p.requires_grad and not any(x in n for x in nd)], "weight_decay":weight_decay},
        {"params":[p for n,p in model.named_parameters()
                   if p.requires_grad and any(x in n for x in nd)], "weight_decay":0.0},
    ], lr=lr)

def _forward(model, b):
    name = type(model).__name__
    if name=="RobertaVAModel":
        return model(b["input_ids"],b["attention_mask"],b["user_idx"],
                     b["prev_v"],b["prev_a"],b["log_delta"],
                     b["is_words_f"],b["is_first"],b["phase_idx"])
    if name=="DebertaVAModel":
        return model(b["input_ids"],b["attention_mask"],b["token_type_ids"],
                     b["user_idx"],b["prev_v"],b["prev_a"],b["log_delta"],
                     b["is_words_f"],b["is_first"],b["phase_idx"])
    raise ValueError(name)

def _to_dev(b):
    out={}
    for k,v in b.items():
        if torch.is_tensor(v):
            out[k]=v.to(DEVICE,dtype=torch.float32) if v.is_floating_point() else v.to(DEVICE)
        else: out[k]=v
    return out

def train_epoch(model, loader, optimizer, scheduler=None, loss_mode="mse"):
    model.train(); total=0.0
    for batch in loader:
        b=_to_dev(batch); optimizer.zero_grad()
        vp,ap=_forward(model,b)
        loss=va_loss(vp, ap, b["valence"], b["arousal"], mode=loss_mode)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        optimizer.step()
        if scheduler is not None: scheduler.step()
        total+=loss.item()
    return total/len(loader)

@torch.no_grad()
def validate(model, loader):
    model.eval(); vp_a,ap_a,vt_a,at_a,uids,tids=[],[],[],[],[],[]
    for batch in loader:
        b=_to_dev(batch); vp,ap=_forward(model,b)
        vp_a+=vp.cpu().tolist(); ap_a+=ap.cpu().tolist()
        vt_a+=b["valence"].cpu().tolist(); at_a+=b["arousal"].cpu().tolist()
        uids+=b["user_id_str"]; tids+=b["text_id_str"]
    return {"valence":task1_correlation(uids,tids,np.array(vp_a),np.array(vt_a)),
            "arousal":task1_correlation(uids,tids,np.array(ap_a),np.array(at_a))}

@torch.no_grad()
def _pred_std(model, loader):
    model.eval(); pv,pa=[],[]
    for batch in loader:
        b=_to_dev(batch); vp,ap=_forward(model,b)
        pv+=vp.cpu().tolist(); pa+=ap.cpu().tolist()
    return float(np.std(pv)), float(np.std(pa))

@torch.no_grad()
def export_predictions(model, df, tokenizer, out_path, batch_size=16):
    model.eval()
    ld=DataLoader(VADataset(df,tokenizer,has_labels=False),
                  batch_size=batch_size,shuffle=False,num_workers=2)
    uids,tids,pv,pa=[],[],[],[]
    for batch in ld:
        b=_to_dev(batch); vp,ap=_forward(model,b)
        pv+=vp.cpu().tolist(); pa+=ap.cpu().tolist()
        uids+=b["user_id_str"]; tids+=b["text_id_str"]
    out=pd.DataFrame({"user_id":uids,"text_id":tids,"pred_valence":pv,"pred_arousal":pa})
    out.to_csv(out_path,index=False); print("saved",out_path,len(out)); return out

## 5. Experiment

In [ ]:
train_df = pd.read_csv(f"{DATA}/train_a_df_all.csv")
train_df = train_df[train_df["text"].notna() &
                    train_df["text"].apply(lambda x: isinstance(x,str)) &
                    (train_df["text"].str.strip()!="")].reset_index(drop=True)

uid_list=sorted(train_df["user_id"].unique())
uid2idx={u:i+1 for i,u in enumerate(uid_list)}
n_users=len(uid_list)

raw_tr, raw_va = temporal_user_split(train_df, val_ratio=0.2)
df_tr=build_temporal_features(raw_tr, raw_tr)
df_va=build_temporal_features(raw_va, raw_tr)
for d in (df_tr,df_va):
    d["user_idx_mapped"]=d["user_id"].map(uid2idx).fillna(0).astype(int)

test_df=pd.read_csv(f"{DATA}/test_subtask1.csv")
df_te=build_temporal_features_test(test_df, train_df)
df_te["user_idx_mapped"]=df_te["user_id"].map(uid2idx).fillna(0).astype(int)

tok_r=RobertaTokenizer.from_pretrained("FacebookAI/roberta-large")
tok_d=DebertaV2Tokenizer.from_pretrained("microsoft/deberta-v3-large")
SEEDS=[42,123,456]
print(f"train={len(df_tr)} val={len(df_va)} test={len(df_te)} users={n_users}")

### 1. RoBERTa

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def train_one_seed(make_model, tokenizer, seed, epochs=10, lr=3e-5, save_path=None):
    set_seed(seed)
    tr_ld=DataLoader(VADataset(df_tr,tokenizer),batch_size=16,shuffle=True,
                     num_workers=2,pin_memory=True)
    va_ld=DataLoader(VADataset(df_va,tokenizer),batch_size=16,shuffle=False,
                     num_workers=2,pin_memory=True)
    model=make_model().to(DEVICE); opt=make_optimizer(model,lr)
    best_score,best_state=-1e9,None
    for ep in range(epochs):
        loss=train_epoch(model,tr_ld,opt)
        m=validate(model,va_ld)
        score=(m["valence"]["r_composite"]+m["arousal"]["r_composite"])/2
        print(f"  seed{seed} ep{ep+1:02d} loss={loss:.4f} score={score:.4f} "
              f"V={m['valence']['r_composite']:.3f} A={m['arousal']['r_composite']:.3f}")
        if score>best_score:
            best_score=score
            best_state={k:v.cpu().clone() for k,v in model.state_dict().items()}
    if save_path:
        torch.save({"model_state":best_state,"uid2idx":uid2idx,"n_users":n_users,"seed":seed},
                   save_path)
    print(f"  seed{seed} BEST={best_score:.4f}")
    del model; torch.cuda.empty_cache(); return best_score

for seed in SEEDS:
    ck=f"{MDL}/rich_roberta_seed{seed}.pt"
    if os.path.exists(ck): print(f"roberta seed{seed} ada, skip."); continue
    train_one_seed(lambda: RobertaVAModel(n_users=n_users), tok_r, seed,
                   epochs=10, save_path=ck)

### 2. DeBERTa

In [ ]:
def ccc_loss(pred, target, eps=1e-8):
    pm, tm = pred.mean(), target.mean()
    vp, vt = pred.var(unbiased=False), target.var(unbiased=False)
    cov = ((pred - pm) * (target - tm)).mean()
    return 1 - 2*cov / (vp + vt + (pm - tm)**2 + eps)

def va_loss(vp, ap, vt, at, mode="mse"):
    if mode == "mse":
        mse = F.mse_loss
        return 0.4*mse(vp, vt) + 0.6*mse(ap, at)          # perilaku lama
    # ccc: korelasi (lawan kolaps) + MAE kecil (jangkar skala), bobot 0.7:0.3
    ccc = 0.4*ccc_loss(vp, vt) + 0.6*ccc_loss(ap, at)
    mae = 0.4*F.l1_loss(vp, vt) + 0.6*F.l1_loss(ap, at)
    return 0.7*ccc + 0.3*mae

In [ ]:
DEB_LR=3e-5; WARMUP_FRAC=0.08; COLLAPSE_TH=0.05; EARLY_PATIENCE=3
def train_deberta_seed(seed, epochs=10, save_path=None):
    set_seed(seed)
    tr_ld = DataLoader(VADataset(df_tr, tok_d), batch_size=16, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)   # drop_last bantu varians CCC
    va_ld = DataLoader(VADataset(df_va, tok_d), batch_size=16, shuffle=False,
                       num_workers=2, pin_memory=True)
    model = DebertaVAModel(n_users=n_users).to(DEVICE)
    opt   = make_optimizer(model, DEB_LR)
    total = len(tr_ld) * epochs
    sched = get_cosine_schedule_with_warmup(opt, int(WARMUP_FRAC*total), total)
    best_score, best_state = -1e9, None
    for ep in range(epochs):
        loss = train_epoch(model, tr_ld, opt, sched, loss_mode="ccc")   # <-- CCC
        m  = validate(model, va_ld)
        sv, sa = _pred_std(model, va_ld)
        score = (m["valence"]["r_composite"] + m["arousal"]["r_composite"]) / 2
        print(f"  seed{seed} ep{ep+1:02d} loss={loss:.4f} score={score:.4f} "
              f"V={m['valence']['r_composite']:.3f} A={m['arousal']['r_composite']:.3f} "
              f"| std V={sv:.3f} A={sa:.3f}")
        if score > best_score and sa > COLLAPSE_TH:
            best_score = score
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    if best_state is None:
        print(f"  seed{seed} GAGAL (selalu kolaps) — turunkan LR lagi atau cek data")
        del model; torch.cuda.empty_cache(); return None
    if save_path:
        torch.save({"model_state": best_state, "uid2idx": uid2idx, "n_users": n_users,
                    "seed": seed, "val_score": best_score}, save_path)
    print(f"  seed{seed} BEST(val)={best_score:.4f}")
    del model; torch.cuda.empty_cache()
    return best_score

deb_val_scores = {}
for seed in SEEDS:
    ck = f"{MDL}/seeds_deberta_seed{seed}.pt"
    if os.path.exists(ck): print(f"deberta seed{seed} ada, skip."); continue
    s = train_deberta_seed(seed, epochs=10, save_path=ck)
    if s is not None: deb_val_scores[seed] = s

### Predict

In [ ]:
def load_roberta(seed):
    model=RobertaVAModel(n_users=n_users).to(DEVICE)
    ck=torch.load(f"{MDL}/rich_roberta_seed{seed}.pt",map_location=DEVICE,weights_only=False)
    model.load_state_dict(ck["model_state"]); model.eval(); return model

def load_deberta_seed(seed):
    model = DebertaVAModel(n_users=n_users).to(DEVICE)
    ck = torch.load(f"{MDL}/seeds_deberta_seed{seed}.pt", map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model_state"]); model.eval(); return model

for seed in SEEDS:
    out=f"{RES}/rich_roberta_seed{seed}_test_predictions.csv"
    if os.path.exists(out): print(f"roberta seed{seed} preds ada, skip."); continue
    m=load_roberta(seed); export_predictions(m,df_te,tok_r,out)
    del m; torch.cuda.empty_cache()

for seed in SEEDS:
    out = f"{RES}/seeds_deberta_seed{seed}_test_predictions.csv"
    if os.path.exists(out): print(f"deberta seed{seed} preds ada, skip."); continue
    m = load_deberta_seed(seed); export_predictions(m, df_te, tok_d, out)
    del m; torch.cuda.empty_cache()

### Ensemble

In [ ]:
def load_preds_path(path):
    df=pd.read_csv(path)
    df["user_id"]=df["user_id"].astype(str).str.extract(r"(\d+)").astype(int)
    df["text_id"]=df["text_id"].astype(str).str.extract(r"(\d+)").astype(int)
    return df.sort_values(["user_id","text_id"]).reset_index(drop=True)

In [ ]:
# Weighted Average

def ensemble_paths(paths, weights=None):
    base = load_preds_path(paths[0])[["user_id","text_id"]].copy()
    dfs  = [load_preds_path(p) for p in paths]
    if weights is None:
        weights = np.ones(len(paths))                 # fallback = uniform (perilaku lama)
    w = np.clip(np.asarray(weights, float), 0, None)  # negatif -> 0
    w = w / (w.sum() + 1e-9)
    vs = sum(wi * d["pred_valence"].values for wi, d in zip(w, dfs))
    az = sum(wi * d["pred_arousal"].values for wi, d in zip(w, dfs))
    base["pred_valence"] = vs
    base["pred_arousal"] = az
    return base

def get_val_scores(ckpt_paths):
    out = []
    for p in ckpt_paths:
        ck = torch.load(p, map_location="cpu", weights_only=False)
        out.append(ck.get("val_score", ck.get("fold_score", 1.0)))  # fallback aman
    return out

roberta_ckpts = [f"{MDL}/rich_roberta_seed{s}.pt"  for s in SEEDS]
deberta_ckpts = [f"{MDL}/seeds_deberta_seed{s}.pt"   for s in SEEDS]
roberta_paths = [f"{RES}/rich_roberta_seed{s}_test_predictions.csv"      for s in SEEDS]
deberta_paths = [f"{RES}/seeds_deberta_seed{s}_test_predictions.csv"       for s in SEEDS]

rob_w = get_val_scores(roberta_ckpts)     # otomatis dari validasi, bukan ketik tangan
deb_w = get_val_scores(deberta_ckpts)

ens = {
    "roberta_ens": ensemble_paths(roberta_paths, rob_w),
    "deberta_ens": ensemble_paths(deberta_paths, deb_w),
    "full_ens":    ensemble_paths(roberta_paths + deberta_paths, rob_w + deb_w),
}
for name, df in ens.items():
    df.to_csv(f"{RES}/{name}_test_predictions.csv", index=False); print("saved", name, len(df))

In [ ]:
test_gold=pd.read_csv(f"{DATA}/test_labels_subtask1.csv")
test_gold["user_id"]=test_gold["user_id"].astype(str).str.extract(r"(\d+)").astype(int)
test_gold["text_id"]=test_gold["text_id"].astype(str).str.extract(r"(\d+)").astype(int)

def score(pred):
    df=pred if isinstance(pred,pd.DataFrame) else load_preds_path(pred)
    m=df.merge(test_gold[["user_id","text_id","valence","arousal"]],
               on=["user_id","text_id"],how="inner")
    v=task1_correlation(m["user_id"],m["text_id"],m["pred_valence"],m["valence"])
    a=task1_correlation(m["user_id"],m["text_id"],m["pred_arousal"],m["arousal"])
    return v,a

rows=[]
for s in SEEDS:
    v,a=score(f"{RES}/rich_roberta_seed{s}_test_predictions.csv")
    rows.append({"model":f"roberta_seed{s}","V_comp":v["r_composite"],"V_within":v["r_within"],
                 "V_between":v["r_between"],"A_comp":a["r_composite"],"A_within":a["r_within"],
                 "A_between":a["r_between"]})
for s in SEEDS:
    v,a=score(f"{RES}/seeds_deberta_seed{s}_test_predictions.csv")
    rows.append({"model":f"deberta_member{s}","V_comp":v["r_composite"],"V_within":v["r_within"],
                 "V_between":v["r_between"],"A_comp":a["r_composite"],"A_within":a["r_within"],
                 "A_between":a["r_between"]})
for name,df in ens.items():
    v,a=score(df)
    rows.append({"model":name,"V_comp":v["r_composite"],"V_within":v["r_within"],
                 "V_between":v["r_between"],"A_comp":a["r_composite"],"A_within":a["r_within"],
                 "A_between":a["r_between"]})
res=pd.DataFrame(rows).round(4)
res["mean_comp"]=((res["V_comp"]+res["A_comp"])/2).round(4)
res=res.sort_values("mean_comp",ascending=False)
res.to_csv(f"{RES}/all_models_scores.csv",index=False)
display(res)

### Post Processing

In [ ]:
rom scipy.optimize import minimize_scalar
from sklearn.isotonic import IsotonicRegression

# kalibrator
def fit_temperature(pc, gc):
    return minimize_scalar(lambda T: _mae(pc/T, gc), bounds=(0.05,20.0), method="bounded").x
def fit_affine(pc, gc):   return np.polyfit(pc, gc, 1)
def fit_isotonic(pc, gc):
    iso = IsotonicRegression(out_of_bounds="clip"); iso.fit(pc, gc); return iso
def categorize_valence(v):
    return 2.0 if v>0.6 else 1.0 if v>0.35 else 0.0 if v>0.15 else -1.0 if v>-0.35 else -2.0
def categorize_arousal(a):
    return 2.0 if a>0.0 else 1.0 if a>-0.4 else 0.0
